1. Setup & Config
2. Load Counterfactual Data
3. Prompt Definition (ONE place only)
4. LLM Clients (unified interface)
5. Sampling (2 CFs per model)
6. Generation Loop
7. Save Outputs
8. Summary / Preview

### 📦 1. Setup, Load API & Config

In [1]:
import os
import json
import time
import numpy as np
import pandas as pd

from dotenv import load_dotenv
import os

load_dotenv("all_api.env", override=True)

# sanity check (prints first few chars only)
def check_key(name):
    val = os.getenv(name)
    return val[:8] + "..." if val else None

print("GROQ:", check_key("GROQ_API_KEY"))
print("OPENAI:", check_key("OPENAI_API_KEY"))
print("GEMINI:", check_key("GEMINI_API_KEY"))
print("MOONSHOT:", check_key("MOONSHOT_API_KEY"))

# Paths
CF_PATH = "artifacts/counterfactuals.csv"
OUTPUT_DIR = "artifacts/meals"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Models
LLM_MODELS = {
    "groq_llama": "llama-3.3-70b-versatile",
    "gemini": "gemini-2.5-flash",
    "openai": "gpt-5.4-mini",
    #"moonshot": "moonshotai/kimi-k2-instruct-0905",
}

N_CF_PER_MODEL = 2
RANDOM_STATE = 42

GROQ: gsk_8Oap...
OPENAI: sk-proj-...
GEMINI: AIzaSyBs...
MOONSHOT: gsk_8Oap...


### 📂 2. Load Counterfactual Data

In [2]:
df = pd.read_csv(CF_PATH)

mask = df[['carbs_cf', 'protein_cf', 'fat_cf']].notna().any(axis=1)
df = df[mask].copy()

print("Total CF rows:", len(df))
df.head()

Total CF rows: 50


,status,y_pred_orig,row_idx,y_true,y_pred_cf,carbs_orig,carbs_cf,protein_orig,protein_cf,fat_orig,fat_cf
3,ok,145.654381,3,138.000000,138.966990,70.0,69.993584,0.0,0.000542,30.0,29.849990
6,ok,145.002197,6,139.206087,139.433966,51.0,44.648342,27.5,27.489235,21.7,21.733252
9,ok,163.413077,9,145.255752,145.406133,99.0,17.146850,32.1,7.080364,37.1,25.984531
11,ok,148.263513,11,211.515892,140.022048,25.7,17.101419,0.0,1.080159,0.0,9.053237
21,ok,158.619787,21,152.000000,140.453044,82.7,15.537950,29.4,25.089339,29.0,26.044081


### 🧾 3. Defining the Prompt

In [3]:
SYSTEM_PROMPT = """
You are a clinical nutrition planning assistant. Your job is to convert target macronutrients into ONE realistic single-meal recommendation using ONLY USDA FoodData Central foods or official USDA food entries.
You must optimize for four goals in this order:
1) clinical plausibility and meal realism,
2) adherence to target macronutrients,
3) nutritional balance and variety,
4) simplicity.

A valid output must look like a real meal that a person could reasonably eat. Do NOT output a snack, a random food pile, or a minimal macro-only plate.

Hard constraints:
- Use 3 to 5 food items whenever possible.
- Every meal should include:
  - 1 main protein source,
  - 1 carbohydrate source,
  - 1 non-starchy vegetable or fruit,
  - 0 to 1 added fat source or garnish.
- Avoid repeating the same “default fillers” across meals, especially almonds, unless they are clearly the best fit and used in a small supporting amount.
- Do not rely on nuts as the main way to satisfy fat targets.
- Do not produce meals with only meat + nuts, or only protein + tiny vegetable portions.
- Keep portions within realistic household serving sizes.
- Use standard servings whenever possible, and avoid tiny “token” amounts such as 2 tbsp beans or similarly implausible portions unless the target macros truly require it.
- Avoid meals that are extremely low in carbohydrate unless the target itself is low-carb and the meal remains realistic.
- Include flavor and palatability: if needed, choose reasonable combinations that people actually eat.
- Prefer minimally processed whole foods.
- If the dietary constraint prevents a normal meal structure, explain the limitation briefly in the notes field.

Macro rules:
- First try to match all target macros within ±10%.
- If exact matching is not feasible while still producing a realistic meal, relax the target in this order:
  1) carbs,
  2) fat,
  3) protein.
- Protein should generally not fall below target unless impossible.
- Do not over-correct by collapsing carbs to near zero when the target is moderate or high.
- Preserve a balanced distribution rather than forcing extreme macro minimization.
- If the requested macro targets imply an implausible meal, return the closest realistic meal and clearly state why in notes.

Food selection rules:
- Prefer foods that naturally fit together as a meal:
  examples: chicken + brown rice + broccoli; salmon + quinoa + vegetables; yogurt + fruit + oats; beans + rice + vegetables.
- Use USDA FoodData Central entries only.
- For each item, choose the closest USDA match and include the FDC name and FDC ID when available.
- When multiple options are similar, prefer the one that improves meal realism and balance, not just macro precision.
- Do not choose bizarre pairings solely because they solve macros.

Output requirements:
- Return ONLY valid JSON.
- Include a brief meal-level rationale in notes.
- Include a feasibility flag so the caller can see whether the target was matched cleanly or only approximately.
- Include a meal-level structure check indicating whether the meal contains protein, carbohydrate, and produce.

JSON schema:
{
  "usda_sources": [
    {
      "name": "<USDA name>",
      "fdc_id": "<optional>",
      "source_url": "<optional>",
      "approximation_note": "<optional if a closest match was used>"
    }
  ],
  "meal_items": [
    {
      "name": "<food name>",
      "fdc_id": "<optional>",
      "amount_g": <grams numeric>,
      "household_measure": "<e.g. 1 cup, 1 medium apple, 4 oz>",
      "carbs_g": <float>,
      "protein_g": <float>,
      "fat_g": <float>,
      "calories_kcal": <float>,
      "role": "<protein|carb|vegetable|fruit|fat|other>"
    }
  ],
  "total_carbs_g": <float>,
  "total_protein_g": <float>,
  "total_fat_g": <float>,
  "total_calories_kcal": <float>,
  "constraints": {
    "target_carbs_g": <float>,
    "target_protein_g": <float>,
    "target_fat_g": <float>,
    "target_calories_kcal": <float or null>,
    "allowed_deviation_percent": <float>,
    "dietary_constraints": "<string>"
  },
  "meal_quality_checks": {
    "has_protein_source": <true/false>,
    "has_carbohydrate_source": <true/false>,
    "has_non_starchy_produce": <true/false>,
    "has_reasonable_portions": <true/false>,
    "uses_repeated_filler_ingredients": <true/false>
  },
  "notes": "<brief note about any compromises, substitutions, or approximation>"
}

USER CONTEXT:
- Target macros: carbs = $target_carbs_g g, protein = $target_protein_g g, fat = $target_fat_g g.
- Optional calorie target: $target_calories_kcal kcal (or null).
- Dietary constraints: $dietary_constraints (e.g., vegetarian, no dairy, none).

PROCESS TO FOLLOW:
1) Build a meal concept first: protein + carb + produce.
2) Search USDA items that fit the concept.
3) Check whether the meal still looks like an actual meal.
4) Check whether the portions are normal and edible.
5) Only then finalize the macro fit.

Finish: Return only the JSON object described above.
"""

### 🧑‍🍳 4. User Prompt Builder

In [4]:
def build_user_prompt(row):
    return f"""
Target macros:
- carbs = {row['carbs_cf']} g
- protein = {row['protein_cf']} g
- fat = {row['fat_cf']} g
- calories = {row.get('calories_cf', None)}

Dietary constraints: {row.get('dietary_constraints', 'none')}
"""

### 🤖 4. LLM Clients (unified)

In [5]:
import os
from openai import OpenAI


def call_groq(model, system_prompt, user_prompt):
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY missing (check all_api.env)")

    from groq import Groq
    client = Groq(api_key=api_key)

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )

    return completion.choices[0].message.content


def call_openai(model, system_prompt, user_prompt):
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY missing (check all_api.env)")

    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
    )

    return response.output[0].content[0].text


def call_gemini(model, system_prompt, user_prompt, max_retries=5):
    import time
    import random
    import os
    from google import genai

    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("GEMINI_API_KEY missing")

    client = genai.Client(api_key=api_key)

    prompt = system_prompt + "\n\n" + user_prompt

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt,
            )

            text = response.text.strip()

            # clean markdown if present
            if text.startswith("```"):
                text = text.split("```")[1]

            return text

        except Exception as e:
            err_str = str(e)

            # Handle 503 specifically
            if "503" in err_str or "UNAVAILABLE" in err_str:
                wait_time = min(2 ** attempt + random.uniform(0, 1), 10)
                print(f"Gemini 503 — retrying in {wait_time:.1f}s (attempt {attempt})")
                time.sleep(wait_time)
                continue

            # other errors → fail fast
            return json.dumps({"error": err_str})

    return json.dumps({"error": "Max retries exceeded for Gemini"})


### 🎯 5. Sampling (2 CFs per model)

In [6]:
def sample_cf(df):
    return df.sample(n=N_CF_PER_MODEL, random_state=RANDOM_STATE)

sampled_df = sample_cf(df)

print("Sampled rows:")
sampled_df[["row_idx", "carbs_cf", "protein_cf", "fat_cf"]]

Sampled rows:


,row_idx,carbs_cf,protein_cf,fat_cf
69,69,1.216793,35.553496,20.846082
103,103,17.147997,1.549246,5.935023


### 🚀 6. Run Generation

In [7]:
def run_model(name, model_name, df):
    print(f"\nRunning {name}...")

    outputs = []

    for _, row in df.iterrows():
        prompt = build_user_prompt(row)

        try:
            if name == "groq_llama":
                resp = call_groq(model_name, SYSTEM_PROMPT, prompt)
            
            elif name == "moonshot": 
                resp = call_groq(model_name, SYSTEM_PROMPT, prompt)
            
            elif name == "openai":
                resp = call_openai(model_name, SYSTEM_PROMPT, prompt)
            
            elif name == "gemini":
                resp = call_gemini(model_name, SYSTEM_PROMPT, prompt)
            else:
                continue

            try:
                parsed = json.loads(resp)
            except:
                parsed = {"raw": resp, "parse_error": True}

        except Exception as e:
            parsed = {"error": str(e)}

        outputs.append({
            "row_idx": int(row["row_idx"]),
            "model": name,
            "response": parsed
        })

        print(f"Row {row['row_idx']} done")

    return outputs

### 🔁 Run All Models

In [8]:
all_outputs = []

for name, model_name in LLM_MODELS.items():
    outputs = run_model(name, model_name, sampled_df)

    out_path = f"{OUTPUT_DIR}/{name}_meals.csv"
    pd.DataFrame(outputs).to_csv(out_path, index=False)

    print(f"Saved → {out_path}")

    all_outputs.extend(outputs)


Running groq_llama...
Row 69 done
Row 103 done
Saved → artifacts/meals/groq_llama_meals.csv

Running gemini...
Row 69 done
Row 103 done
Saved → artifacts/meals/gemini_meals.csv

Running openai...
Row 69 done
Row 103 done
Saved → artifacts/meals/openai_meals.csv


### 💾 7. Save Combined Output

In [9]:
all_df = pd.DataFrame(all_outputs)

all_df.to_csv(f"{OUTPUT_DIR}/all_meals.csv", index=False)

print("Saved combined results")

Saved combined results


### 📊 8. Preview Results

In [10]:
all_df.head()

,row_idx,model,response
0,69,groq_llama,"{'usda_sources': [{'name': 'Chicken Breast', '..."
1,103,groq_llama,"{'usda_sources': [{'name': 'Chicken Breast', '..."
2,69,gemini,"{'raw': 'json { ""usda_sources"": [ { ..."
3,103,gemini,"{'raw': 'json { ""usda_sources"": [ { ..."
4,69,openai,"{'usda_sources': [{'name': 'Turkey breast, roa..."
